In [ ]:
import pandas as pd

from os import path
import re

from pymodulon.io import load_json_model
from pymodulon.plotting import *

import numpy as np
import plotly.express as px
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['svg.fonttype'] = 'none'
matplotlib.rcParams['font.sans-serif'] = 'Arial'
matplotlib.rcParams['font.family'] = 'sans-serif'
sns.set_style('ticks')
matplotlib.rcParams['text.color'] = '#000000'
matplotlib.rcParams['axes.labelcolor'] = '#000000'
matplotlib.rcParams['xtick.color'] = '#000000'
matplotlib.rcParams['ytick.color'] = '#000000'

In [ ]:
ica_data = load_json_model('../../modulome_sc.json')

In [ ]:
A_condition = pd.DataFrame(index=ica_data.A.index, columns=ica_data.sample_table.full_name.unique())

for condition in ica_data.sample_table.full_name.unique():
    inds = ica_data.sample_table[ica_data.sample_table.full_name == condition].index
    mean = ica_data.A[inds].mean(axis=1)
    A_condition[condition] = mean

In [ ]:
metabolic_ims = list(ica_data.imodulon_table[ica_data.imodulon_table.category == 'Metabolism'].index)
cellular_process_ims = list(ica_data.imodulon_table[ica_data.imodulon_table.category == 'Cellular Processes'].index)
stress_ims = list(ica_data.imodulon_table[ica_data.imodulon_table.category == 'Stress'].index)
unchar_ims = list(ica_data.imodulon_table[ica_data.imodulon_table.index.str.contains('Unchar')].index)

In [ ]:
ica_data.A.T.corr().loc['HAP4'].sort_values()

In [ ]:
g = sns.clustermap(A_condition.loc[metabolic_ims].T.corr(), cmap='coolwarm', center = 0, figsize = (8,8))
g.fig.suptitle('Metabolic iModulon Activity Clustermap', y= 1.02, fontsize = 25)

plt.savefig('images/metabolic_ims_activity_clustermap.svg', format = 'svg')

In [ ]:
im = "HAP4"
for ind in ica_data.view_imodulon(im).gene_weight.index:
    print(ind, '\t', ica_data.view_imodulon(im).gene_weight.loc[ind])

In [ ]:
ica_data.view_imodulon('HAP4').sort_values('gene_weight')

In [ ]:
compare_gene_weights(ica_data, 'HAP4', 'Low Glucose Response')

In [ ]:
hap4_genes = ica_data.view_imodulon('HAP4').index
lgr_genes = ica_data.view_imodulon('Low Glucose Response').index

shared_genes = set(hap4_genes).intersection(set(lgr_genes))


other_genes = [gene for gene in ica_data.X if gene not in hap4_genes and gene not in lgr_genes]


groups_hap4 = {}
groups_lgr = {}
for gene in ica_data.X.index:
    if gene in shared_genes:
        groups_hap4[gene] = 'Shared Gene'
        groups_lgr[gene] = 'Shared Gene'
    elif gene in hap4_genes:
        groups_hap4[gene] = 'Gene in HAP4 iM'
        groups_lgr[gene] = 'Other Genes'
    elif gene in lgr_genes:
        groups_hap4[gene] = 'Other Genes'
        groups_lgr[gene] = 'Gene in LGR iM'
    else:
        groups_hap4[gene] = 'Other Genes'
        groups_lgr[gene] = 'Other Genes'


colors = {'Shared Gene':'Green', 'Gene in HAP4 iM': 'tab:orange', 'Gene in LGR iM': 'tab:orange', 'Other Genes': 'Grey'}

In [ ]:
im  = 'HAP4'
plot_gene_weights(ica_data, im, show_labels = False, groups = groups_hap4, colors = colors)
plt.legend(loc='lower right')
plt.savefig('images/fig3_hap4_gene_weights.svg', format = 'svg')

In [ ]:
im  = 'Low Glucose Response'
plot_gene_weights(ica_data, im, groups = groups_lgr, colors = colors)
plt.legend(loc='upper right')
plt.savefig('images/fig3_lgr_gene_weights.svg', format = 'svg')

In [ ]:
control_starvation = [i for i,x in ica_data.sample_table[ica_data.sample_table.project == 'CLS_ETC_mutants'].full_name.items() if "wt_t0" in x]
mid_starvation = [i for i,x in ica_data.sample_table[ica_data.sample_table.project == 'CLS_ETC_mutants'].full_name.items() if "wt_t12" in x]
late_starvation = [i for i,x in ica_data.sample_table[ica_data.sample_table.project == 'CLS_ETC_mutants'].full_name.items() if "wt_t16" in x]
df_diff = dima(ica_data, control_starvation, mid_starvation, threshold=20, fdr=0.01)


norm = matplotlib.colors.TwoSlopeNorm(vmin=df_diff['difference'].min(),
                            vcenter=0,
                            vmax=df_diff['difference'].max())

# Map colors from the palette using the normalized values
cmap = plt.get_cmap('seismic')
colors = [cmap(norm(val)) for val in df_diff['difference']]


fig, ax = plt.subplots(1,1, figsize=(10,6))
sns.barplot(df_diff, x=df_diff.index, y='difference', ax=ax, palette=colors)
plt.title('Differential iModulon Activity Between Starvation Samples at 0h and 12h')
plt.ylabel('Activity Difference')
ax.tick_params(axis='x', rotation=0)
plt.savefig('images/starvation_dima_barplot.svg', format = 'svg')
df_diff

In [ ]:
imod1 = "HAP4"
imod2 = 'Low Glucose Response'

cls_etc = ica_data.sample_table[ica_data.sample_table.project == 'CLS_ETC_mutants'].copy()
cls_etc[imod1] = ica_data.A.loc[imod1, list(cls_etc.index)]
cls_etc[imod2] = ica_data.A.loc[imod2, list(cls_etc.index)]
cls_etc['time'] = cls_etc.condition.apply(lambda x: x.split(':')[-1].split('_')[-1])

ebu_histone = ica_data.sample_table[ica_data.sample_table.project == 'ebu_histone_acetyltransferase'].copy()
ebu_histone[imod1] = ica_data.A.loc[imod1, list(ebu_histone.index)]
ebu_histone[imod2] = ica_data.A.loc[imod2, list(ebu_histone.index)]
ebu_histone['name'] = ebu_histone.condition.apply(lambda x: x.replace('pde', 'pds'))
ebu_histone['metabolic status'] = ebu_histone['name'].apply(lambda x: x.split('_')[-1])

# Plotting
fig, ax = plt.subplots(figsize=(10, 8))

# Background scatter
x_acts, y_acts = ica_data.A.loc[imod1], ica_data.A.loc[imod2]
ax.scatter(x_acts, y_acts, c='tab:grey', s=5, alpha=0.1)


cls_plot = sns.scatterplot(
    data=cls_etc, x=imod1, y=imod2,
    hue='time', palette='tab20', ax=ax
)
# Get handles/labels for CLS_ETC only
handles1, labels1 = ax.get_legend_handles_labels()
ax.legend_.remove()

ebu_plot = sns.scatterplot(
    data=ebu_histone, x=imod1, y=imod2,
    hue='metabolic status', palette=['Purple', 'Green'], ax=ax,
    marker = 's'
)
handles2_all, labels2_all = ax.get_legend_handles_labels()


existing_labels_set = set(labels1)
handles2_raw = list(zip(handles2_all, labels2_all))
handles2, labels2 = zip(*[
    (h, l) for h, l in handles2_raw
    if l not in existing_labels_set
])


square_handles = [
    Line2D([0], [0], marker='s', color='w', label=label,
           markerfacecolor=h.get_facecolor()[0], markersize=6, markeredgecolor='black')
    for h, label in zip(handles2, labels2)
]


legend1 = ax.legend(handles1, labels1, title="Starvation Dataset: \nTime", loc='upper left')
legend2 = ax.legend(square_handles, labels2, title="Post Diauxic-Shift: \nMetabolic Status", loc='lower right')
ax.add_artist(legend1)


plt.xlabel('HAP4 iM Activity')
plt.ylabel('Low Glucose Response iM Activity')
plt.tight_layout()
plt.savefig('images/fig3_im_activities_scatter.svg', format = 'svg')
plt.show()

In [ ]:
off_diagonal_conditions = ica_data.A.loc[['HAP4', 'Low Glucose Response'], 
               (ica_data.A.loc['HAP4'] > 25) & 
               (ica_data.A.loc['Low Glucose Response'] < 5)].columns

ica_data.sample_table.loc[off_diagonal_conditions]

In [ ]:
ica_data.A.loc[['HAP4', 'Low Glucose Response'],off_diagonal_conditions].mean(axis=1)

In [ ]:
off_diagonal_conditions = ica_data.A.loc[['HAP4', 'Low Glucose Response'], 
               (ica_data.A.loc['HAP4'] > 30) & 
               (ica_data.A.loc['Low Glucose Response'] < 20)].columns

ica_data.sample_table.loc[off_diagonal_conditions]

# Analysis of Regulation of LGR iModulon

In [ ]:
TRN_ENRICHMENTS = path.join('..', '..','data', 'processed_data', 'trn_enrichments.csv')
TRN_ENRICHMENTS_SUPP = path.join('..', '..','data', 'processed_data', 'trn_enrichments_supp.csv')
KEGG_ENRICHMENTS_PATHWAYS = path.join('..', '..','data', 'processed_data', 'kegg_pathway_enrichments.csv')
KEGG_ENRICHMENTS_MODULES = path.join('..', '..','data', 'processed_data', 'kegg_module_enrichments.csv')
GO_ENRICHMENTS = path.join('..', '..','data', 'processed_data', 'go_enrichment.csv')

trn_enrichments = pd.read_csv(TRN_ENRICHMENTS, index_col=0, low_memory=False)
trn_enrichments_supp = pd.read_csv(TRN_ENRICHMENTS_SUPP, index_col=0, low_memory=False)
kegg_pathway_enrichments = pd.read_csv(KEGG_ENRICHMENTS_PATHWAYS, index_col=0, low_memory=False)
kegg_module_enrichments = pd.read_csv(KEGG_ENRICHMENTS_MODULES, index_col=0, low_memory=False)
go_enrichments = pd.read_csv(GO_ENRICHMENTS, index_col=0, low_memory=False)

In [ ]:
trn = pd.read_csv('../../data/external/trn_full.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)


count = 0
genes_in_regulons = trn["gene_id"].unique()
for gene in ica_data.X.index:
    if gene in genes_in_regulons:
        count+=1
print("Number of genes with annotated regulators: ",count)

trn = trn[trn.gene_id.isin(ica_data.gene_table.index)]

In [ ]:
im = 'Low Glucose Response'
im_number = list(ica_data.imodulon_table.index).index(im)

In [ ]:
import pandas as pd
from upsetplot import from_memberships, plot
import matplotlib.pyplot as plt

trn_im_subset = trn[trn.gene_id.isin(ica_data.view_imodulon(im).index) & trn.regulator.str.contains('MIG')] # subset of TRN in LGR and regulated by MIG genes


gene_reg_groups = trn_im_subset.groupby('gene_id')['regulator'].apply(list)

data_for_upset = from_memberships(gene_reg_groups)

plot(data_for_upset, subset_size='count', show_counts=True)
plt.title('Regulator Overlap for Low Glucose Response Genes')
plt.show()

## Look at LFR Regulation

In [ ]:
X = ica_data.X
trn = pd.read_csv('../../data/external/trn_full.csv', index_col=0).dropna()
trn = trn[["reg","gene_name", "gene_id", "effect"]]
trn = trn.rename({"reg":"regulator"}, axis=1)


count = 0
genes_in_regulons = trn["gene_id"].unique()
for gene in X.index:
    if gene in genes_in_regulons:
        count+=1
print("Number of genes with annotated regulators: ",count)
print("Number of unique regulators:", len(trn.regulator.unique()))
#filter trn to only include genes which were aligned to
display(trn.head(), trn.shape)

In [ ]:
im = 'Low Glucose Response'
im_number = list(ica_data.imodulon_table.index).index(im)

genes_imod = (
    ica_data.view_imodulon(im)
    .sort_values("gene_weight", ascending=False)
    .index
)

top_regs = trn_enrichments_supp[trn_enrichments_supp.imodulon == im_number]["regulator"].tolist()

# Build binary matrix
mat = pd.DataFrame(0, index=top_regs, columns=genes_imod)

for reg in top_regs:
    targets = set(trn[trn["regulator"] == reg]["gene_id"])
    overlap = targets & set(genes_imod)
    mat.loc[reg, list(overlap)] = 1

# Optional: remove genes with no overlap (shouldn’t happen but safe)
mat = mat.loc[:, mat.sum(axis=0) > 0]


from matplotlib.patches import Patch

g = sns.clustermap(
    mat,
    cmap=sns.color_palette(["white", "black"], as_cmap=True),
    figsize=(20, 20),
    row_cluster=False,
    col_cluster=False,
    cbar=False
)

# Completely remove colorbar axis
g.cax.set_visible(False)

# Rotate y-axis labels 90 degrees and increase font size
g.ax_heatmap.set_yticklabels(
    g.ax_heatmap.get_yticklabels(),
    rotation=0,
    fontsize=16
)

# Increase x-axis tick font size (optional)
g.ax_heatmap.set_xticklabels(
    g.ax_heatmap.get_xticklabels(),
    rotation=90,
    fontsize=12
)

# Axis labels
g.ax_heatmap.set_xlabel("Genes in iModulon", fontsize=18)
g.ax_heatmap.set_ylabel("Regulators", fontsize=18)

# Binary legend
legend_elements = [
    Patch(facecolor="black", edgecolor="black", label="Target gene"),
    Patch(facecolor="white", edgecolor="black", label="No target")
]

g.ax_heatmap.legend(
    handles=legend_elements,
    title="Regulatory overlap",
    bbox_to_anchor=(1.05, 1),
    loc="upper left",
    fontsize=14,
    title_fontsize=15
)

plt.suptitle(
    "Regulator–Gene Overlap Heatmap",
    fontsize=22,
    y=1.02
)

plt.savefig(
    "images/fig3_lgr_regulator_heatmap.svg",
    format="svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib as mpl

df = trn_enrichments_supp[
    trn_enrichments_supp.imodulon == im_number
]

fig, ax = plt.subplots(figsize=(8,4))

scatter = sns.scatterplot(
    data=df,
    x="recall",
    y="precision",
    size="regulon_size",
    hue="precision",
    sizes=(50,500),
    palette="Reds",
    ax=ax,
    legend="brief"
)

# Extract the scatter object created by seaborn
points = ax.collections[0]

# Create a ScalarMappable for the colorbar
norm = mpl.colors.Normalize(
    vmin=df["precision"].min(),
    vmax=df["precision"].max()
)

sm = mpl.cm.ScalarMappable(
    cmap="Reds",
    norm=norm
)
sm.set_array([])

# Add colorbar while keeping size legend
cbar = fig.colorbar(
    sm,
    ax=ax,
    pad=0.02
)

cbar.set_label("Fraction of iModulon in Regulon")

# Add labels
for _, row in df.iterrows():
    ax.text(
        row.recall,
        row.precision,
        row.regulator,
        fontsize=9
    )

ax.set_xlabel("Fraction of Regulon captured")
ax.set_ylabel("Fraction of iModulon in Regulon")

ax.set_title(
    "Regulatory Architecture of Low Glucose Response iModulon"
)

# plt.tight_layout()
plt.savefig('images/fig3_lgr_regulator_precision_recall.svg', format = 'svg')
plt.show()